# CineScope Phase 1 — ResNet-18 shot-scale classifier on MovieShots

Settings: **Accelerator: GPU**, **Internet: on**. Input: your private `cinescope-code` dataset.
MovieShots is shared by its authors as a Google Drive folder
(movienet.github.io/projects/eccv20shot.html); cell 2 downloads it with `gdown`.
If Google Drive refuses (download quota), download it yourself, upload it as a private
Kaggle dataset, attach it, and re-run: cell 2 uses an attached copy first.

Two runs: ImageNet-initialised (30 epochs) and from scratch (60 epochs). Use
**Save & Run All (Commit)**; if 12 h is not enough for both, run the second in a new version.

In [ ]:
# 1. Find the code: the "cinescope-code" dataset (Kaggle unpacks the zip) -> a writable copy
import glob, os, shutil, subprocess, sys
hits = glob.glob("/kaggle/input/**/p1_cnn/train_framing.py", recursive=True)
assert hits, "Add the cinescope-code dataset to this notebook (Add Input -> Your Datasets)"
src = os.path.dirname(os.path.dirname(hits[0]))
shutil.copytree(src, "/kaggle/working/cinescope", dirs_exist_ok=True)
os.chdir("/kaggle/working/cinescope")
print("code from", src)
!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h /kaggle/working /tmp | tail -n 2

In [ ]:
# 2. MovieShots: attached copy, or download from the authors' Google Drive folder
!pip -q install gdown pytest
import json
MS = None
for j in glob.glob("/kaggle/input/**/*.json", recursive=True):
    if "trailer" in os.path.basename(j).lower():
        MS = os.path.dirname(j); break
if MS is None:
    MS = "/tmp/movieshots"
    subprocess.run(["gdown", "--folder", "https://drive.google.com/drive/folders/1wAviJV0fuISKG6RpCAqiwG7lwtfxktkQ",
                    "-O", MS, "--remaining-ok"], check=True)
    for z in glob.glob(f"{MS}/**/*.zip", recursive=True) + glob.glob(f"{MS}/**/*.tar*", recursive=True):
        print("unpacking", z)
        if z.endswith(".zip"):
            subprocess.run(["unzip", "-q", "-o", z, "-d", os.path.dirname(z)], check=True)
        else:
            subprocess.run(["tar", "-xf", z, "-C", os.path.dirname(z)], check=True)
        os.remove(z)
# show the layout (the index builder below reports anything it cannot use)
for root, dirs, files in os.walk(MS):
    depth = root[len(MS):].count(os.sep)
    if depth <= 2:
        print("  " * depth + os.path.basename(root) + "/", f"({len(files)} files)", files[:3])

In [ ]:
# 3. Build the flat index (trailer-level splits), then tests + smoke run
json_files = glob.glob(f"{MS}/**/*split*.json", recursive=True) or glob.glob(f"{MS}/**/*.json", recursive=True)
ROOT = os.path.dirname(json_files[0])
!python p1_cnn/prepare_movieshots.py --root {ROOT} --annotations {json_files[0]} --out /kaggle/working/index.csv
!python -m pytest -q tests/test_week4.py tests/test_week5.py
!python p1_cnn/train_framing.py --synthetic --epochs 1 --out /tmp/smoke --ckpt-dir /tmp/smoke

In [ ]:
# 4. ImageNet-initialised run
!python p1_cnn/train_framing.py --root {ROOT} --index /kaggle/working/index.csv --pretrained --epochs 30 \
    --tag pretrained --out /kaggle/working/results --ckpt-dir /kaggle/working/checkpoints --workers 4

In [ ]:
# 5. From-scratch run (the comparison the roadmap asks for)
!python p1_cnn/train_framing.py --root {ROOT} --index /kaggle/working/index.csv --epochs 60 \
    --tag scratch --out /kaggle/working/results --ckpt-dir /kaggle/working/checkpoints --workers 4

In [ ]:
!cd /kaggle/working && zip -qr phase1_results.zip results checkpoints index.csv && ls -lh phase1_results.zip